Modularizing the code means breaking down the code into small pieces, we will be following that.

In [0]:

process_name = dbutils.widgets.get("prm_processName")
process_status = "Completed"

next_source_file_date = f"""select max(processed_file_table_date)+1 as next_source_file_date from processrunlogs.deltalakehouse_process_runs
where process_name = '{process_name}' and process_status = "Completed";"""

next_source_file_date = spark.sql(next_source_file_date)

next_source_file_date.select("next_source_file_date").collect()[0]['next_source_file_date'] #SQL array needs to be converted to python string variable which will then be passed to the date variable to pull from HTTP source

What we have done is to create a sql table for logging every run which we can use to pull the date for the most recent run and use that as an input for the next file to be pulled. What we can also do is use the parameter as a source which pulls the data from another notebook or a source which can also be used as an input date to run the file. What also needs to be done is to put a check to make sure the pipeline does not fail where you can say the date the not more than today's date or yesterday's date depending on business use case.


In [0]:
dbutils.widgets.help()

In [0]:
from datetime import datetime
dbutils.widgets.text("prm_dailyPricingSourceFileDate","")
print((datetime.strptime(str(next_source_file_date.select("next_source_file_date").collect()[0]['next_source_file_date']),"%Y-%m-%d")).strftime("%d%m%Y"))#dbutils accepts only string format input



In [0]:
source_file_url = "https://retailpricing.blob.core.windows.net/labs/lab1/PW_MW_DR_01012023.csv"

##source_file_ingestion_path = "abfss://working-labs@datalakestorageaccountname.dfs.core.windows.net/bronze/daily-pricing/csv"

In [0]:
daily_pricing_source_base_url = "https://retailpricing.blob.core.windows.net/"
daily_pricing_source_folder = "daily-pricing/"
daily_pricing_source_file = "PW_MW_DR_01012023.csv"
daily_pricing_source_file_date = (datetime.strptime(str(next_source_file_date.select("next_source_file_date").collect()[0]['next_source_file_date']),"%Y-%m-%d")).strftime("%d%m%Y")
daily_pricing_source_file_name = f"PW_MW_DR_{daily_pricing_source_file_date}.csv"

daily_pricing_sink_layer_name = "adbbronze"
daily_pricing_storage_account = "adbstorageu"
daily_pricing_sink_folder = "daily-pricing"

In [0]:

print(daily_pricing_source_file_name)

In [0]:
import pandas as pd

In [0]:
daily_pricing_source_url = daily_pricing_source_base_url + daily_pricing_source_folder + daily_pricing_source_file_name

In [0]:
print(daily_pricing_source_url)

In [0]:
daily_pricing_pd_df = pd.read_csv(daily_pricing_source_url)
print(daily_pricing_pd_df.head())

In [0]:
daily_pricing_spark_df = spark.createDataFrame(daily_pricing_pd_df)

In [0]:
%python
# Set the storage account key
spark.conf.set(
    f"fs.azure.account.key.{daily_price_storage_account}.dfs.core.windows.net",
    "uQj8Bupjr0W/zcdBLh/FIFZ300z/wnPoLqmRvsl6fu/vdvebCYK1BVekNIHEYNFC2jyIU1jCW2Nc+ASt1Jij0w=="
)

In [0]:
daily_pricing_sink_folder = f"abfss://{daily_pricing_sink_layer_name}@{daily_pricing_storage_account}.dfs.core.windows.net/{daily_price_sink_folder}"

##daily_price_sink_folder = "abfss://bronze@adbstorageu.dfs.core.windows.net/daily-pricing"

print(daily_pricing_sink_folder)

(
    daily_pricing_spark_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(daily_pricing_sink_folder)
)

In [0]:
##Duplicated loads can run, we need to ensure that we are not duplicating the data.f]'c

In [0]:
%sql
USE CATALOG adb_rtp;

CREATE Schema if not exists processrunlogs;

CREATE Table if not exists processrunlogs.deltalakehouse_process_runs(process_name string,
  processed_file_table_date Date,
  process_status string
);

In [0]:
process_name = "daily_pricing_ingest"
process_file_table_date = dbutils.widgets.get("prm_dailyPricingSourceFileDate")
process_status = "Completed"


In [0]:
process_insert_SQL = f""" INSERT INTO processrunlogs.deltalakehouse_process_runs VALUES ('{process_name}','{process_file_table_date}','{process_status}')"""

spark.sql(process_insert_SQL) ##Remember this, to run a SQL statement in a notebook, you need to use spark.sql()

In [0]:
%sql

select * from processrunlogs.deltalakehouse_process_runs;

In [0]:
%sql 
select max(processed_file_table_date)+1 as next_source_file_date from processrunlogs.deltalakehouse_process_runs as next_source_file_date
where process_name = "daily_pricing_ingest" and process_status = "Completed";